# 05 - Baseline architecture
Find out which model the CAD2 Task 2 baseline uses, describe its architecture, and compare a few variants on the valid set.

**Before this, run `00_shared_drive_data.ipynb`:** the clone-Clarity cell and **Part A** (builds the valid-set dataset). It puts Clarity in `/content/clarity` and the dataset in `/content/cadenza_data/cad2/task2`.

Note: this notebook uses paths relative to `/content`, so a Colab session restart means running 00 again. Save results to Drive (`MyDrive/Cadenza Challenge/results/`) because free Colab can disconnect.

1. Check prerequisites (nothing is downloaded here)
Stops with a clear message if notebook 00 has not been run in this session.

In [ ]:
import os
from pathlib import Path
os.chdir('/content')

root = Path('/content/cadenza_data/cad2/task2')
clarity_dir = Path('/content/clarity')
meta = root / 'metadata'
baseline = clarity_dir / 'recipes/cad2/task2/baseline'

need = {
    'Clarity baseline (run 00 clone cell or 01_setup)': baseline / 'enhance.py',
    'valid metadata (run 00 Part A)': meta / 'scenes.valid.json',
    'valid audio (run 00 Part A)': root / 'audio/valid',
}
missing = [k for k, p in need.items() if not p.exists()]
assert not missing, 'Missing: ' + '; '.join(missing)
print('prerequisites ok')

## Part A - Find the baseline model (Saturday)

2. Read the baseline recipe
Show `config.yaml` and `enhance.py`, then find where the model is loaded: which class, which file it comes from, and the causal / non-causal switch.

In [ ]:
!grep -n "ConvTasNet\|from_pretrained\|import\|causal" /content/clarity/recipes/cad2/task2/baseline/enhance.py

!cat /content/clarity/recipes/cad2/task2/baseline/config.yaml

3:from __future__ import annotations
5:import json
6:import logging
7:from collections import Counter
8:from pathlib import Path
10:# pylint: disable=import-error
11:import hydra
12:import numpy as np
13:import torch
14:from numpy import ndarray
15:from omegaconf import DictConfig
16:from torchaudio.transforms import Fade
18:from clarity.enhancer.multiband_compressor import MultibandCompressor
19:from clarity.utils.audiogram import Listener
20:from clarity.utils.flac_encoder import read_flac_signal, save_flac_signal
21:from clarity.utils.source_separation_support import get_device
22:from recipes.cad2.task2.baseline.evaluate import (
28:from recipes.cad2.task2.ConvTasNet.local.tasnet import ConvTasNetStereo
128:    model: dict[str, ConvTasNetStereo],
175:    causality: str, device: torch.device, force_redownload: bool = True
176:) -> dict[str, ConvTasNetStereo]:
180:        causality (str): Causality of the model (causal or noncausal).
188:    causal = {"causal": "Causal", "noncausal":

3. List the models Cadenza published on HuggingFace
Which instruments have a model, and are there Causal as well as NonCausal versions?

In [19]:
# list models from the cadenzachallenge HuggingFace account
from huggingface_hub import list_models
for m in list_models(author='cadenzachallenge'):
    print(m.id)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


cadenzachallenge/ConvTasNet_Lyrics_NonCausal
cadenzachallenge/ConvTasNet_Lyrics_Causal
cadenzachallenge/ConvTasNet_Bassoon_NonCausal
cadenzachallenge/ConvTasNet_Cello_NonCausal
cadenzachallenge/ConvTasNet_Clarinet_NonCausal
cadenzachallenge/ConvTasNet_Flute_NonCausal
cadenzachallenge/ConvTasNet_Oboe_NonCausal
cadenzachallenge/ConvTasNet_Sax_NonCausal
cadenzachallenge/ConvTasNet_Viola_NonCausal
cadenzachallenge/ConvTasNet_Violin_NonCausal
cadenzachallenge/ConvTasNet_Bassoon_Causal
cadenzachallenge/ConvTasNet_Cello_Causal
cadenzachallenge/ConvTasNet_Clarinet_Causal
cadenzachallenge/ConvTasNet_Flute_Causal
cadenzachallenge/ConvTasNet_Oboe_Causal
cadenzachallenge/ConvTasNet_Sax_Causal
cadenzachallenge/ConvTasNet_Viola_Causal
cadenzachallenge/ConvTasNet_Violin_Causal
cadenzachallenge/Dynamic_Source_Separator_Causal
cadenzachallenge/Dynamic_Source_Separator_Noncausal
cadenzachallenge/CLIP2-BaselineMono
cadenzachallenge/CLIP2-BaselineBetterEar


4. Load one model and describe it
Print the structure, parameter count and config.

In [ ]:

# TODO: load one pretrained model, print structure / parameter count / config

5. Architecture table
Fill in using the Conv-TasNet paper's notation, and compare with `third_party/Conv-TasNet/conv_tasnet.py`.

| | Cadenza baseline | naplab Conv-TasNet (third_party) |
|---|---|---|
| N (encoder filters) | | 512 |
| L (window) | | 2 ms |
| B (bottleneck) | | 128 |
| H (hidden) | | |
| P (kernel) | | |
| X (blocks per repeat) | | 8 |
| R (repeats) | | 3 |
| Causal | | |
| Sample rate | | 16 kHz |
| Mono / stereo | | |
| Outputs (sources) | | |
| Parameters | | |

6. Where the model sits in the pipeline
mixture -> one separator per instrument -> scene gains -> remix -> NAL-R + compressor (listener) -> output (32 kHz)

Notes: which parts are learned and which are fixed signal processing?

## Part B - Comparison numbers (Sunday)

7. Pick valid scenes
About 5 scenes, each with its first listener. Each scene takes roughly 4 min to enhance plus 1 min to evaluate.

In [ ]:
# TODO: choose ~5 valid scenes and write a small scenes / scene_listeners file

8. Comparison A - NonCausal (default) vs Causal
HAAQI left / right / avg per scene. If there are no Causal models, compare against applying the gains to the whole mixture with no separation.

In [ ]:
# TODO: run enhance.py + evaluate.py for each variant, collect scores.csv

9. Comparison B - separation quality
SDR of each separated stem against its ground-truth stem, for 1-2 scenes.

In [ ]:
# TODO: per-stem SDR

10. Save results to Drive

In [ ]:
# TODO: write the comparison CSVs to /content/drive/MyDrive/Cadenza Challenge/results/

## Summary
- Baseline model:
- Key differences from the original Conv-TasNet:
- Comparison results:
- Questions for Kirill:
- Next step (retraining): see `recipes/cad2/task2/ConvTasNet/` in Clarity (check that it exists)